# 05 — Treinamento dos modelos

Treinamento de três redes simples para classificar os espectrogramas (já com CLAHE + contraste, em `mod_spectograms`) nas quatro classes: `ictal`, `normal`, `pos_ictal` e `pre_ictal`. A divisão treino/validação segue o `splits.csv`.

1. **CNN**: convoluções + camadas densas.
2. **FCNN** (Fully Convolutional Network): apenas convoluções, sem camadas densas.
3. **Encoder-Decoder**: comprime a imagem (encoder) e depois a reconstrói (decoder) antes de classificar.

A avaliação no conjunto de teste está no notebook `06_metrics.ipynb`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout,
                                     Conv2DTranspose, GlobalAveragePooling2D, Activation)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
tf.keras.utils.set_random_seed(0)
print(tf.config.list_physical_devices('GPU'))

I0000 00:00:1795029001.547314   10509 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795029001.547855   10509 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1795029001.579932   10509 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


[]


I0000 00:00:1795029002.888580   10509 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795029002.888969   10509 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
E0000 00:00:1795029002.982827   10509 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## Carregamento dos dados

In [2]:
splits = pd.read_csv(PROJECT_ROOT / 'data' / 'annotations' / 'splits.csv')
classes = sorted(splits['classe'].unique())
print(classes)

def carregar(conjunto):
    df = splits[splits['conjunto'] == conjunto]
    X, y = [], []
    for _, row in df.iterrows():
        caminho = PROJECT_ROOT / row['caminho'].replace('spectrograms', 'mod_spectograms')
        img = Image.open(caminho).convert('RGB').resize((224, 224))
        X.append(np.array(img) / 255.0)
        y.append(classes.index(row['classe']))
    return np.array(X, dtype='float32'), np.array(y)

X_train, y_train = carregar('treino')
X_val, y_val = carregar('validacao')
print(X_train.shape, X_val.shape)

def plotar_historico(history, titulo):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, metrica, nome in zip(axes, ['loss', 'accuracy'], ['Perda', 'Acurácia']):
        ax.plot(history.history[metrica], label='Treinamento')
        ax.plot(history.history['val_' + metrica], label='Validação')
        ax.set_title(f'{titulo} — {nome}')
        ax.set_xlabel('Épocas')
        ax.set_ylabel(metrica)
        ax.legend()
    plt.tight_layout()
    plt.show()

['ICTAL', 'NORMAL', 'POS_ICTAL', 'PRE_ICTAL']
(100, 224, 224, 3) (20, 224, 224, 3)


# 1. CNN

## Arquitetura

In [3]:
model = Sequential()
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3))) # Convolucao 2D
model.add(MaxPooling2D((2, 2))) # Camada de coarsen
model.add(Conv2D(64, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))
model.add(Conv2D(128, (3, 3), activation='relu'))
model.add(MaxPooling2D((2, 2)))

model.add(Flatten())

model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(4, activation='softmax'))

model.summary()

/home/omp/Documents/senai/epilepsy-detection/.venv/lib/python3.10/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,476 (42.61 MB)

 Trainable params: 11,169,476 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

## Treinamento

In [4]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history = model.fit(X_train, y_train, epochs=30, validation_data=(X_val, y_val), verbose=2)

Epoch 1/30
4/4 - 2s - 423ms/step - accuracy: 0.2000 - loss: 1.7457 - val_accuracy: 0.3000 - val_loss: 1.3738
Epoch 2/30
4/4 - 1s - 229ms/step - accuracy: 0.3500 - loss: 1.3618 - val_accuracy: 0.2500 - val_loss: 1.3725
Epoch 3/30
4/4 - 1s - 231ms/step - accuracy: 0.4000 - loss: 1.3596 - val_accuracy: 0.4500 - val_loss: 1.3117
Epoch 4/30
4/4 - 1s - 238ms/step - accuracy: 0.3900 - loss: 1.3313 - val_accuracy: 0.4000 - val_loss: 1.2450
Epoch 5/30
4/4 - 1s - 247ms/step - accuracy: 0.4400 - loss: 1.2951 - val_accuracy: 0.3500 - val_loss: 1.2236
Epoch 6/30
4/4 - 1s - 249ms/step - accuracy: 0.3600 - loss: 1.2737 - val_accuracy: 0.3500 - val_loss: 1.1665
Epoch 7/30
4/4 - 1s - 246ms/step - accuracy: 0.4600 - loss: 1.1658 - val_accuracy: 0.4000 - val_loss: 1.1678
Epoch 8/30
4/4 - 1s - 249ms/step - accuracy: 0.6100 - loss: 1.0141 - val_accuracy: 0.4000 - val_loss: 1.1221
Epoch 9/30
4/4 - 1s - 238ms/step - accuracy: 0.5700 - loss: 0.9784 - val_accuracy: 0.4500 - val_loss: 1.2109
Epoch 10/30
4/4 - 1

In [5]:
print(f"Acurácia final de treinamento: {history.history['accuracy'][-1]:.4f}")
print(f"Acurácia final de validação:   {history.history['val_accuracy'][-1]:.4f}")
print(f"Loss final de treinamento:     {history.history['loss'][-1]:.4f}")
print(f"Loss final de validação:       {history.history['val_loss'][-1]:.4f}")

Acurácia final de treinamento: 0.9900
Acurácia final de validação:   0.3500
Loss final de treinamento:     0.0630
Loss final de validação:       2.7182


## Gráficos

In [11]:
plotar_historico(history, 'CNN')

## Resultados (CNN)

A acurácia de treino chega perto de 100%, enquanto a de validação fica entre 35% e 45%, e a loss de validação sobe ao longo das épocas. Isso indica overfitting: a rede tem cerca de 11 milhões de parâmetros para apenas 100 imagens de treino. Com só 20 imagens de validação, cada imagem vale 5 pontos percentuais, o que explica a oscilação da curva.

## Salvando o modelo

In [9]:
model.save(PROJECT_ROOT / 'models' / 'modelo_cnn.keras')

## Perguntas (CNN)

**O que faz o otimizador Adam?**
Ele ajusta os pesos da rede a partir do gradiente da loss, mas com uma taxa de aprendizado adaptativa para cada peso. Combina o momentum (média dos gradientes anteriores, que suaviza a direção da atualização) com o RMSProp (média dos gradientes ao quadrado, que regula o tamanho do passo). Na prática converge rápido e funciona bem sem muito ajuste de hiperparâmetros.

**Por que a camada final utiliza Softmax?**
Porque é um problema multiclasse com 4 classes mutuamente exclusivas. A softmax transforma as 4 saídas em probabilidades que somam 1, e a classe prevista é a de maior probabilidade. Isso também casa com a `sparse_categorical_crossentropy`, que espera uma distribuição de probabilidade na saída.

**Quantas camadas convolucionais existem?**
3 (`Conv2D` com 32, 64 e 128 filtros).

**Quantas camadas de pooling existem?**
3 (`MaxPooling2D` 2x2 após cada convolução).

**Quantos parâmetros possui a rede?**
11.169.476 parâmetros treináveis:

| Camada | Saída | Parâmetros |
|---|---|---|
| Conv2D (32) | 222x222x32 | 3·3·3·32 + 32 = 896 |
| MaxPooling2D | 111x111x32 | 0 |
| Conv2D (64) | 109x109x64 | 3·3·32·64 + 64 = 18.496 |
| MaxPooling2D | 54x54x64 | 0 |
| Conv2D (128) | 52x52x128 | 3·3·64·128 + 128 = 73.856 |
| MaxPooling2D | 26x26x128 | 0 |
| Flatten | 86.528 | 0 |
| Dense (128) | 128 | 86.528·128 + 128 = 11.075.712 |
| Dropout | 128 | 0 |
| Dense (4) | 4 | 128·4 + 4 = 516 |

Quase tudo (99%) está na primeira camada densa, por causa do Flatten de 86.528 valores.

**Qual camada é responsável pela classificação final?**
A última `Dense(4, activation='softmax')`, que gera a probabilidade de cada uma das 4 classes.

# 2. FCNN (Fully Convolutional Network)

Rede formada apenas por convoluções. No lugar do `Flatten` + `Dense`, uma convolução 1x1 gera 4 mapas (um por classe) e o `GlobalAveragePooling2D` tira a média de cada mapa, resultando em 4 valores que vão para a Softmax. Por não ter camadas densas, ela tem muito menos parâmetros que a CNN.

## Arquitetura

In [ ]:
fcnn = Sequential([
    Input(shape=(224, 224, 3)),
    Conv2D(16, (3, 3), activation='relu', padding='same'),
    MaxPooling2D((2, 2)),
    Conv2D(32, (3, 3), activation='relu', padding='same'),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu', padding='same'),
    MaxPooling2D((2, 2)),
    Conv2D(4, (1, 1)),          # 1 mapa por classe
    GlobalAveragePooling2D(),   # média de cada mapa -> 4 valores
    Activation('softmax'),
])

fcnn.summary()

## Treinamento

In [ ]:
fcnn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_fcnn = fcnn.fit(X_train, y_train, epochs=30, validation_data=(X_val, y_val), verbose=2)

## Gráficos

In [ ]:
plotar_historico(history_fcnn, 'FCNN')

## Resultados (FCNN)

Observe nos gráficos se a distância entre treino e validação é menor que na CNN. Com bem menos parâmetros (~24 mil contra ~11 milhões), a FCNN tende a decorar menos o treino, mas também aprende mais devagar.

## Salvando o modelo

In [ ]:
fcnn.save(PROJECT_ROOT / 'models' / 'modelo_fcnn.keras')

# 3. Encoder-Decoder

- **Encoder**: convoluções + pooling reduzem a imagem de 224x224 para 56x56, extraindo as características principais.
- **Decoder**: convoluções transpostas (`Conv2DTranspose`) aumentam a imagem de volta para 224x224.
- **Classificação**: `GlobalAveragePooling2D` + `Dense` com Softmax nas 4 classes.

## Arquitetura

In [ ]:
encdec = Sequential([
    Input(shape=(224, 224, 3)),
    # Encoder
    Conv2D(16, (3, 3), activation='relu', padding='same'),
    MaxPooling2D((2, 2)),                                                       # 112x112
    Conv2D(32, (3, 3), activation='relu', padding='same'),
    MaxPooling2D((2, 2)),                                                       # 56x56
    # Decoder
    Conv2DTranspose(32, (3, 3), strides=2, activation='relu', padding='same'),  # 112x112
    Conv2DTranspose(16, (3, 3), strides=2, activation='relu', padding='same'),  # 224x224
    # Classificação
    GlobalAveragePooling2D(),
    Dense(4, activation='softmax'),
])

encdec.summary()

## Treinamento

In [ ]:
encdec.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_encdec = encdec.fit(X_train, y_train, epochs=30, validation_data=(X_val, y_val), verbose=2)

## Gráficos

In [ ]:
plotar_historico(history_encdec, 'Encoder-Decoder')

## Resultados (Encoder-Decoder)

Observe nos gráficos se a validação acompanha o treino. O decoder devolve a imagem ao tamanho original antes da classificação, então a rede trabalha em alta resolução e o treino é o mais lento dos três modelos.

## Salvando o modelo

In [ ]:
encdec.save(PROJECT_ROOT / 'models' / 'modelo_encdec.keras')

# Comparação das arquiteturas

| Modelo | Camadas convolucionais | Camadas densas | Parâmetros |
|---|---|---|---|
| CNN | 3 `Conv2D` | 2 `Dense` | 11.169.476 |
| FCNN | 4 `Conv2D` (a última 1x1) | nenhuma | 23.844 |
| Encoder-Decoder | 2 `Conv2D` + 2 `Conv2DTranspose` | 1 `Dense` | 19.028 |

Quase todos os parâmetros da CNN estão na camada `Dense` depois do `Flatten`. Trocar essa parte por `GlobalAveragePooling2D` (FCNN e Encoder-Decoder) reduz o modelo em mais de 400 vezes.

A comparação das métricas no conjunto de teste está no notebook `06_metrics.ipynb`.